In [66]:
%load_ext autoreload
%load_ext line_profiler
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler


In [67]:
import pandas as pd
from krxns.network import ReactionNetwork
from ergochemics.draw import draw_reaction
from IPython.display import SVG
from pathlib import Path

In [68]:
G = ReactionNetwork.from_json('/home/stef/krxns/data/processed/known_reaction_network.json')
print("Full reaction network loaded from JSON.")
print(f"Number of nodes: {G.number_of_nodes()}, Number of edges: {G.number_of_edges()}")
default_sources = pd.read_csv('/home/stef/krxns/data/interim/default_sources.csv')['smiles'].tolist()


Full reaction network loaded from JSON.
Number of nodes: 9253, Number of edges: 87418


In [69]:
i = G.get_nodes_by_prop('name', 'pyruvate')[0]
print(f"Node ID for pyruvate: {i}")
j = G.get_nodes_by_prop('smiles', 'CC(O)C(=O)O')[0]
print(f"Node ID for lactic acid: {j}")

Node ID for pyruvate: 2efe03c1b1595e5fedce6a3674dad82075142215
Node ID for lactic acid: 79d6e08f6a3697ab702833506f6ec9f5e8d6aaab


In [70]:
len(list(G.successors(i)))

139

In [71]:
G.get_edge_data(i, j)

{'c6cd5805f1490515656048ad2d1cd397fdb38237': {'pnmc': 1.0,
  'rnmc': 1.0,
  'am_smarts': '[NH2:43][C:42](=[O:44])[C:6]1=[CH:7][N:2]([CH:1]2[O:10][CH:11]([CH2:12][O:14][P:16](=[O:17])([OH:18])[O:19][P:20](=[O:21])([OH:22])[O:23][CH2:24][CH:25]3[O:26][CH:28]([n:31]4[cH:33][n:35][c:37]5[c:39]([NH2:41])[n:40][cH:38][n:36][c:34]54)[CH:29]([OH:32])[CH:27]3[OH:30])[CH:13]([OH:15])[CH:8]2[OH:9])[CH:3]=[CH:4][CH2:5]1.[CH3:47][C:45](=[O:46])[C:48](=[O:49])[OH:50]>>[NH2:43][C:42](=[O:44])[c:6]1[cH:5][cH:4][cH:3][n+:2]([CH:1]2[O:10][CH:11]([CH2:12][O:14][P:16](=[O:17])([OH:18])[O:19][P:20](=[O:21])([OH:22])[O:23][CH2:24][CH:25]3[O:26][CH:28]([n:31]4[cH:33][n:35][c:37]5[c:39]([NH2:41])[n:40][cH:38][n:36][c:34]54)[CH:29]([OH:32])[CH:27]3[OH:30])[CH:13]([OH:15])[CH:8]2[OH:9])[cH:7]1.[CH3:47][CH:45]([OH:46])[C:48](=[O:49])[OH:50]'},
 '33bba368952d5218fee358a14247116286f279c8': {'pnmc': 1.0,
  'rnmc': 1.0,
  'am_smarts': '[NH2:47][C:46](=[O:48])[C:6]1=[CH:7][N:2]([CH:1]2[O:10][CH:11]([CH2:12][O:14][P:1

In [72]:
kcs = pd.read_parquet('/home/stef/krxns/data/raw/known_compounds.parquet')
krs = pd.read_parquet('/home/stef/krxns/data/raw/mapped_known_reactions_x_rc_plus_0_rules.parquet')
kcs.head()

,id,smiles,name,chebi_id,n_atoms
0,0,*,A,CHEBI:13193,1
1,1,**,RX,CHEBI:17792,2
2,2,*C,an alkane,CHEBI:18310,2
3,3,*C#N,a nitrile,CHEBI:18379,3
4,4,*C(*)(O)C(*)(*)O,an ethanediol,CHEBI:140594,8


In [73]:
kcs[kcs['name'].str.contains('2-ethyl-2-hydroxy-3-oxobutanoate')]

,id,smiles,name,chebi_id,n_atoms
4208,4208,CCC(O)(C(C)=O)C(=O)O,(S)-2-ethyl-2-hydroxy-3-oxobutanoate,CHEBI:49256,10


In [74]:
addtl_sources = {
    'lactate': 'CC(O)C(=O)O',
    'threonine': 'CC(O)C(N)C(=O)O'
}

targets = {'2-ethyl-2-hydroxy-3-oxobutanoate': 'CCC(O)(C(C)=O)C(=O)O'}

In [75]:
sources = default_sources + list(addtl_sources.values())
G.set_sources(smiles=sources)
rnmc_lb = 0.25
pnmc_lb = 0.25
aug_mc_lb = 0.8
# G.prune(pnmc_lb=pnmc_lb, rnmc_lb=rnmc_lb, source_augmented_pnmc_lb=aug_mc_lb)

Set 23 source compounds in the reaction network.


In [76]:
print(f"Pruned network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Pruned network: 9253 nodes, 87418 edges


In [77]:
target = G.get_nodes_by_prop('smiles', targets['2-ethyl-2-hydroxy-3-oxobutanoate'])[0]

In [78]:
# %lprun -f G.enumerate_synthetic_trees G.enumerate_synthetic_trees(target=1844, max_depth=2, max_leaves=3, tot_rnmc_lb=0.1)

In [79]:
trees = G.enumerate_synthetic_trees(target=target, max_depth=2, max_leaves=3, tot_rnmc_lb=0.1)
print(f"Found {len(trees)} synthetic trees.")

Considered 1758446 trees, found 3 synthetic trees.
Found 3 synthetic trees.


In [80]:
for i, tree in enumerate(trees):
    print(f"Tree {i}:")
    for leaf in tree.leaves:
        print(leaf[0], G.nodes[leaf[0]]['name'])
    print("\n")

Tree 0:
79d6e08f6a3697ab702833506f6ec9f5e8d6aaab (R)-lactate
1aa818910461f5961959eb05dd06b8f7a4fbbb9f O2
6daed2b3b791906a7d3848a121a339f1eb68bfdd D-threonine


Tree 1:
da7b7aacec3c6f733a01920b971ac0deda5d3303 NADP(+)
79d6e08f6a3697ab702833506f6ec9f5e8d6aaab (R)-lactate
6daed2b3b791906a7d3848a121a339f1eb68bfdd D-threonine


Tree 2:
599b0cd59ed43643b07bd6f1b3284abcbbb20811 NAD(+)
79d6e08f6a3697ab702833506f6ec9f5e8d6aaab (R)-lactate
6daed2b3b791906a7d3848a121a339f1eb68bfdd D-threonine




In [81]:
trees[0]

SyntheticTree(root='4add7ac7cb340408a0f06bc00c0f1c9625ca6a08', generations=[{'4add7ac7cb340408a0f06bc00c0f1c9625ca6a08': 'a08846ba39a2781a1a016b0066f6ebc0ce3b4eac'}, {'2efe03c1b1595e5fedce6a3674dad82075142215': '336f6705de68d0691e40ec82df8d1a3b309d3cb1', '4c3782cb869285f550e1eca3afe7c436f157a1c3': '75f12d7b2a602a4275b947268d6956a8cf8662c9'}, {'79d6e08f6a3697ab702833506f6ec9f5e8d6aaab': None, '1aa818910461f5961959eb05dd06b8f7a4fbbb9f': None, '6daed2b3b791906a7d3848a121a339f1eb68bfdd': None}], leaves=[('79d6e08f6a3697ab702833506f6ec9f5e8d6aaab', 2), ('1aa818910461f5961959eb05dd06b8f7a4fbbb9f', 2), ('6daed2b3b791906a7d3848a121a339f1eb68bfdd', 2)])

In [51]:
# for i, tree in enumerate(trees):
#     print(f"Tree {i}:")
#     for j, gen in enumerate(tree.generations):
#         print(f"  Generation {j}:")
#         for rid in gen.values():
#             rxn = krs.loc[krs['rxn_id'] == rid, 'smarts'].values[0]
#             display(SVG(draw_reaction(rxn)))

Attempt adding a predicted network

In [52]:
G = ReactionNetwork.from_json('/home/stef/krxns/data/processed/known_reaction_network.json')
exp_dir = "/home/stef/krxns/data/raw/2_steps_bottle_targets_25_to_None_rules_mechinferred_dt_01_rules_w_coreactants_co_metacyc_coreactants_sampled_False_pruned_False_aplusb_False"

In [53]:
with open(Path(exp_dir) / "sources.txt", "r") as f:
    addtl_sources = [line.strip() for line in f.readlines()]

sources = default_sources + addtl_sources
G.set_sources(smiles=sources)

with open(Path(exp_dir) / "targets.txt", "r") as f:
    target_smiles = [line.strip() for line in f.readlines()]


with open(Path(exp_dir) / "am_rxns.txt", "r") as f:
    am_rxns = [line.strip() for line in f.readlines()]

for rxn in am_rxns:
    G.add_reaction(rxn)    

target_ids = [G.get_nodes_by_prop('smiles', smiles)[0] for smiles in target_smiles]

In [54]:
trees = G.enumerate_synthetic_trees(
    target=target_ids[0],
    max_depth=5,
    max_leaves=3,
    tot_rnmc_lb=0.0,
)

Considered 2063 trees, found 0 synthetic trees.


In [55]:
trees

[]

In [56]:
[node for node in G.nodes if node['source']]

TypeError: string indices must be integers, not 'str'

In [ ]:
target_ids[0]

In [ ]:
target1 = G.nodes[target_ids[0]]

In [ ]:
len(target1['tot_rnmc'])

In [ ]:
for k, v in target1['tot_rnmc'].items():
    print(f"{k}: {v}")
    print(target1['grouped_predecessors'][k])
    

In [ ]:
smarts = G.get_edge_data(u='795ce8b8f679817da1f0333c1a7ab70424422dd2', v='6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d', key='8f4821611d805ad2cd0fca8de893e54fa0d8c625')['am_smarts']

In [ ]:
display(SVG(draw_reaction(smarts)))

In [57]:
G.get_edge_data(u='1acaca7bc072c7d623a53bdb8aa450afea98a9b1', v='6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d')

{'d85f833e5c38cacd8947a1698fbd1d01abde2139': {'pnmc': 1.0,
  'rnmc': 0.8888888888888888,
  'am_smarts': '[CH3:1][O:35][C:33](=[O:34])[C:29]([CH3:28])([CH3:30])[CH2:31][OH:32].[NH2:19][c:18]1[n:20][cH:21][n:22][c:23]2[c:17]1[n:16][cH:15][n:14]2[CH:13]1[O:12][CH:11]([CH2:10][S:2][CH2:3][CH2:4][CH:5]([NH2:6])[C:7](=[O:8])[OH:9])[CH:26]([OH:27])[CH:24]1[OH:25]>>[CH3:1][S+:2]([CH2:3][CH2:4][CH:5]([NH2:6])[C:7](=[O:8])[OH:9])[CH2:10][CH:11]1[O:12][CH:13]([n:14]2[cH:15][n:16][c:17]3[c:18]([NH2:19])[n:20][cH:21][n:22][c:23]32)[CH:24]([OH:25])[CH:26]1[OH:27].[CH3:28][C:29]([CH3:30])([CH2:31][OH:32])[C:33](=[O:34])[OH:35]'}}

In [58]:
G.get_edge_data(u='eef113d8ad6695378d0caebf7bae3a67b60c6323', v='6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d')

{'d85f833e5c38cacd8947a1698fbd1d01abde2139': {'pnmc': 0.0,
  'rnmc': 0.0,
  'am_smarts': '[CH3:1][O:35][C:33](=[O:34])[C:29]([CH3:28])([CH3:30])[CH2:31][OH:32].[NH2:19][c:18]1[n:20][cH:21][n:22][c:23]2[c:17]1[n:16][cH:15][n:14]2[CH:13]1[O:12][CH:11]([CH2:10][S:2][CH2:3][CH2:4][CH:5]([NH2:6])[C:7](=[O:8])[OH:9])[CH:26]([OH:27])[CH:24]1[OH:25]>>[CH3:1][S+:2]([CH2:3][CH2:4][CH:5]([NH2:6])[C:7](=[O:8])[OH:9])[CH2:10][CH:11]1[O:12][CH:13]([n:14]2[cH:15][n:16][c:17]3[c:18]([NH2:19])[n:20][cH:21][n:22][c:23]32)[CH:24]([OH:25])[CH:26]1[OH:27].[CH3:28][C:29]([CH3:30])([CH2:31][OH:32])[C:33](=[O:34])[OH:35]'},
 'a50e485ec9587abe3cdc182684997fc865408a8a': {'pnmc': 0.0,
  'rnmc': 0.0,
  'am_smarts': '[NH2:19][c:18]1[n:20][cH:21][n:22][c:23]2[c:17]1[n:16][cH:15][n:14]2[CH:13]1[O:12][CH:11]([CH2:10][S:2][CH2:3][CH2:4][CH:5]([NH2:6])[C:7](=[O:8])[OH:9])[CH:26]([OH:27])[CH:24]1[OH:25].[CH3:1][O:32][CH2:31][C:29]([CH3:28])([CH3:30])[C:33](=[O:34])[OH:35]>>[CH3:1][S+:2]([CH2:3][CH2:4][CH:5]([NH2:6])[C:7

In [60]:
G.nodes['6569ea8ebe9fdd8627adb76f7d942dbbbb83c56d']['tot_rnmc']['d85f833e5c38cacd8947a1698fbd1d01abde2139']

0.22857142857142856

In [ ]:
G.nodes['1acaca7bc072c7d623a53bdb8aa450afea98a9b1']['source']

False

In [64]:
G.nodes['1acaca7bc072c7d623a53bdb8aa450afea98a9b1']['smiles']

'COC(=O)C(C)(C)CO'

In [63]:
G.nodes['eef113d8ad6695378d0caebf7bae3a67b60c6323']['source']

True